# Diabetes Care Information Assistant
Healthcare cognitive assistant for UCS420 Cognitive Computing. My roll number is odd, so this one is the diabetes assistant.

What it does: reads a question, works out what the user wants (the intent), finds the matching info in a small knowledge base, applies a few safety rules and writes a reply.

This is only for general information. It is not medical advice and it can't diagnose anything.

## Setup

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

try:
    import nltk
    for pkg in ["stopwords", "wordnet", "omw-1.4"]:
        nltk.download(pkg, quiet=True)
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    STOPWORDS = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    lemmatizer.lemmatize("tests")
    HAVE_NLTK = True
except Exception:
    STOPWORDS = (set(ENGLISH_STOP_WORDS) - {"low", "high", "fill", "full", "thick", "thin"}) | {"does", "doe", "did", "doing", "okay"}
    lemmatizer = None
    HAVE_NLTK = False

print("NLTK available:", HAVE_NLTK, "| stopwords:", len(STOPWORDS))

## 1. Labelled queries

160 queries, 8 intents, 20 queries each.

In [ ]:
corpus = {
    "diabetes_basics": [
        "what is diabetes",
        "what is the difference between type 1 and type 2 diabetes",
        "explain gestational diabetes",
        "what is prediabetes",
        "why does a person get diabetes",
        "is diabetes a chronic disease",
        "what causes high blood sugar in the long run",
        "who is at risk of developing type 2 diabetes",
        "define type 2 diabetes",
        "what does it mean to be prediabetic",
        "is type 1 diabetes genetic",
        "how common is diabetes in india",
        "can diabetes be cured completely",
        "what is insulin resistance",
        "what is type 1 diabetes",
        "tell me about type 2 diabetes and its causes",
        "what are the types of diabetes",
        "what is the cause of diabetes",
        "what is meant by diabetes mellitus",
        "what are the risk factors for diabetes",
    ],
    "symptoms": [
        "what are the common symptoms of diabetes",
        "why do i feel very thirsty and urinate often",
        "signs of high blood sugar",
        "does diabetes cause blurred vision",
        "i feel tired all the time and lose weight, could it be diabetes",
        "early warning signs of diabetes",
        "why are my cuts healing slowly",
        "what tests are used to diagnose diabetes",
        "what are the warning signs i should look for",
        "does frequent urination mean diabetes",
        "why do my hands and feet tingle",
        "which blood tests confirm diabetes",
        "what is the oral glucose tolerance test",
        "does extreme hunger indicate high sugar",
        "what are the symptoms of type 2 diabetes",
        "what symptoms show that blood sugar is high",
        "symptoms of diabetes in women",
        "which signs tell me i should get tested for diabetes",
        "how is diabetes diagnosed",
        "what is a diagnostic test for diabetes",
    ],
    "diet_nutrition": [
        "what foods should a diabetic eat",
        "which foods should i avoid with diabetes",
        "can diabetic patients eat rice",
        "is fruit safe for people with diabetes",
        "give me a healthy meal plan for diabetes",
        "how many carbohydrates should i eat per meal",
        "are sweets completely banned in diabetes",
        "what snacks are good for blood sugar control",
        "can i drink fruit juice with diabetes",
        "what should i eat for breakfast to control sugar",
        "is sugar free chocolate okay for diabetics",
        "how do i count carbs in my food",
        "is milk or curd good for diabetics",
        "can i eat chapati and dal",
        "what food items should i stay away from",
        "which fruits are good for diabetics",
        "what should a diabetic eat for dinner",
        "is white rice bad for a diabetic",
        "what is a diabetic diet chart",
        "how much sugar can a diabetic eat in food",
    ],
    "medication_insulin": [
        "how does metformin work",
        "what are the different types of insulin",
        "when should insulin be taken",
        "what are the side effects of diabetes tablets",
        "how should i store insulin pens",
        "do all diabetics need insulin injections",
        "what medicines are used for type 2 diabetes",
        "where should i inject insulin",
        "what is the timing for diabetes tablets",
        "does metformin cause stomach upset",
        "what is basal insulin and bolus insulin",
        "can i take my diabetes medicine without food",
        "what are glp-1 injections for diabetes",
        "are insulin injections painful",
        "what are the side effects of insulin",
        "which tablets are prescribed for diabetes",
        "how do insulin injections work",
        "what is the best medicine for diabetes",
        "is metformin safe for long term use",
        "what are the types of diabetes medicines",
    ],
    "glucose_monitoring": [
        "how often should i check my blood sugar",
        "what is a normal fasting blood sugar level",
        "what is hba1c and why does it matter",
        "how do i use a glucometer",
        "what is the target blood glucose range",
        "what is a continuous glucose monitor",
        "what should my sugar be two hours after eating",
        "how often should i test hba1c",
        "what does a high hba1c result mean",
        "what is a normal blood sugar after meals",
        "how to read my glucometer result",
        "when should i test my sugar during the day",
        "what is a good a1c for diabetics",
        "how accurate are finger prick tests",
        "how do i check my glucose at home",
        "what is the normal range for blood sugar",
        "what is the target hba1c for diabetics",
        "how often should i monitor my glucose",
        "what does my hba1c test result mean",
        "what blood sugar reading is considered high",
    ],
    "exercise_lifestyle": [
        "how much exercise should a diabetic do",
        "is walking good for diabetes",
        "can i do yoga to control my sugar",
        "does stress affect blood glucose",
        "how does sleep affect diabetes",
        "should i quit smoking if i have diabetes",
        "can weight loss reverse type 2 diabetes",
        "what precautions should i take before a workout",
        "what exercises are best for diabetics",
        "does running reduce blood sugar",
        "how does being overweight affect my sugar",
        "will drinking alcohol affect my diabetes",
        "can lack of sleep raise blood glucose",
        "how do i manage stress when i have diabetes",
        "is exercise good for diabetes",
        "how long should a diabetic walk daily",
        "what kind of workout is safe for diabetics",
        "does losing weight help diabetes",
        "how does lifestyle change control diabetes",
        "how can sleep and stress affect sugar levels",
    ],
    "complications_prevention": [
        "what complications can diabetes cause",
        "how does diabetes affect the eyes",
        "can diabetes damage the kidneys",
        "what is diabetic neuropathy",
        "how do i take care of my feet with diabetes",
        "how can i prevent diabetic complications",
        "does diabetes increase the risk of heart disease",
        "how often should i get an eye exam",
        "what is diabetic retinopathy",
        "why do diabetics get foot ulcers",
        "how is kidney disease linked to diabetes",
        "can diabetes cause nerve pain",
        "how do i protect my heart if i have diabetes",
        "how often should kidney tests be done",
        "what are the long term complications of diabetes",
        "how does high blood sugar damage the kidneys",
        "how can diabetes harm my eyes and vision",
        "what is diabetic foot and how do i prevent it",
        "can diabetes damage nerves",
        "how do i prevent complications like heart disease and kidney disease",
    ],
    "emergency": [
        "my sugar is very low and i feel dizzy and shaky",
        "what should i do during a hypoglycemia attack",
        "diabetic patient is unconscious",
        "blood sugar is extremely high and i am vomiting",
        "what are the signs of diabetic ketoacidosis",
        "i had a seizure after taking insulin",
        "my father with diabetes is confused and not responding",
        "my breath smells fruity and i feel nauseous",
        "what is the 15 15 rule for low sugar",
        "i am sweating and trembling and my sugar might be low",
        "he fainted and he is diabetic",
        "my sugar is high and i feel drowsy and breathing fast",
        "what is severe hypoglycemia",
        "can i give glucagon to an unconscious person",
        "what do i do when my sugar drops too low",
        "what should i do if a diabetic person faints",
        "signs of low blood sugar emergency",
        "what is diabetic emergency treatment",
        "what to do if blood sugar is dangerously high",
        "how do i treat hypoglycemia quickly",
    ],
}

rows = [(q, intent) for intent, qs in corpus.items() for q in qs]
df = pd.DataFrame(rows, columns=["query", "intent"])
print(f"Total labelled queries: {len(df)} | intents: {df['intent'].nunique()}")
print(df["intent"].value_counts().to_string())
df.sample(8, random_state=1)

## 2. Taxonomy

4 main categories with 2 subcategories each. Every subcategory is tied to one intent.

In [ ]:
taxonomy = {
    "Understanding Diabetes": {
        "Types of Diabetes": "diabetes_basics",
        "Symptoms & Diagnosis": "symptoms",
    },
    "Daily Management": {
        "Diet & Nutrition": "diet_nutrition",
        "Exercise & Lifestyle": "exercise_lifestyle",
    },
    "Treatment & Monitoring": {
        "Medication & Insulin": "medication_insulin",
        "Glucose Monitoring": "glucose_monitoring",
    },
    "Complications & Emergencies": {
        "Long-term Complications": "complications_prevention",
        "Emergency Care": "emergency",
    },
}

print("Diabetes Care Taxonomy")
for i, (cat, subs) in enumerate(taxonomy.items(), 1):
    print(f"{i}. {cat}")
    for sub, intent in subs.items():
        print(f"     - {sub}  [intent: {intent}]")

intent_to_path = {intent: (cat, sub) for cat, subs in taxonomy.items() for sub, intent in subs.items()}
assert len(taxonomy) >= 4 and all(len(s) >= 2 for s in taxonomy.values())

## 3. Concepts and relationships

14 concepts and 13 relationships. The assistant uses these to add a related fact to its answer.

In [ ]:
concepts = {
    "Diabetes":      ["diabetes", "diabetic"],
    "Insulin":       ["insulin"],
    "Blood Glucose": ["blood sugar", "glucose", "sugar level", "sugar"],
    "HbA1c":         ["hba1c", "a1c"],
    "Diet":          ["diet", "food", "meal", "carb", "eat"],
    "Exercise":      ["exercise", "workout", "walking", "yoga"],
    "Hypoglycemia":  ["hypoglycemia", "low sugar", "low blood sugar"],
    "Hyperglycemia": ["hyperglycemia", "high sugar", "high blood sugar"],
    "Metformin":     ["metformin"],
    "Neuropathy":    ["neuropathy", "nerve", "foot", "feet"],
    "Retinopathy":   ["retinopathy", "eye", "vision"],
    "Nephropathy":   ["nephropathy", "kidney"],
    "Pancreas":      ["pancreas"],
    "Obesity":       ["obesity", "overweight", "weight"],
}

relationships = [
    ("Pancreas",      "produces",                    "Insulin"),
    ("Insulin",       "lowers",                      "Blood Glucose"),
    ("Diabetes",      "is characterised by",         "Hyperglycemia"),
    ("HbA1c",         "reflects the 2-3 month average of", "Blood Glucose"),
    ("Diet",          "influences",                  "Blood Glucose"),
    ("Exercise",      "improves the body's response to", "Insulin"),
    ("Metformin",     "is a first-line medicine for type 2", "Diabetes"),
    ("Insulin",       "can cause (if too much or meals are missed)", "Hypoglycemia"),
    ("Hyperglycemia", "if prolonged, damages nerves causing", "Neuropathy"),
    ("Hyperglycemia", "if prolonged, damages the eyes causing", "Retinopathy"),
    ("Hyperglycemia", "if prolonged, damages the kidneys causing", "Nephropathy"),
    ("Obesity",       "increases the risk of type 2", "Diabetes"),
    ("Diet",          "helps prevent", "Obesity"),
]

assert len(concepts) >= 8 and len(relationships) >= 10
print(f"{len(concepts)} concepts, {len(relationships)} relationships\n")
pd.DataFrame(relationships, columns=["Concept A", "Relationship", "Concept B"])

In [ ]:
try:
    import networkx as nx
    G = nx.DiGraph()
    for a, r, b in relationships:
        G.add_edge(a, b, label=r)
    plt.figure(figsize=(11, 7))
    pos = nx.spring_layout(G, seed=7, k=1.1)
    nx.draw_networkx(G, pos, node_color="#cfe8ff", node_size=2600, font_size=8, arrows=True, edge_color="#888")
    plt.title("Diabetes domain concept map")
    plt.axis("off")
    plt.show()
except Exception as e:
    print("Concept map skipped:", e)

## 4. Cleaning the text

Lowercase, split into words, remove stopwords, lemmatize.

In [ ]:
KEEP_AS_IS = {"diabetes", "insulin", "glucose", "gestational", "ketoacidosis", "metformin"}

def simple_lemma(w):
    if w in KEEP_AS_IS or len(w) <= 3:
        return w
    if w.endswith("ies"):
        return w[:-3] + "y"
    if w.endswith("sses"):
        return w[:-2]
    if w.endswith("ing") and len(w) > 5:
        base = w[:-3]
        return base[:-1] if len(base) > 2 and base[-1] == base[-2] else base
    if w.endswith("ed") and len(w) > 4:
        return w[:-2]
    if w.endswith("s") and not w.endswith(("ss", "us", "is")):
        return w[:-1]
    return w

def lemmatize(word):
    if word in KEEP_AS_IS:
        return word
    if HAVE_NLTK:
        return lemmatizer.lemmatize(lemmatizer.lemmatize(word, "n"), "v")
    return simple_lemma(word)

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text)

def preprocess(text, as_tokens=False):
    text = text.lower()
    tokens = tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS]
    tokens = [lemmatize(t) for t in tokens if len(t) > 1 and not t.isdigit()]
    return tokens if as_tokens else " ".join(tokens)

demo = "What are the Symptoms of Diabetes? My feet are tingling and I'm urinating often!"
print("Original :", demo)
print("Lowercase:", demo.lower())
print("Tokens   :", tokenize(demo.lower()))
print("Stopwords removed:", [t for t in tokenize(demo.lower()) if t not in STOPWORDS])
print("Lemmatized:", preprocess(demo, as_tokens=True))

df["clean"] = df["query"].apply(preprocess)
df.head()

## 5. TF-IDF and training the classifiers

Three classifiers are compared with cross-validation and a held-out test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["intent"], test_size=0.25, stratify=df["intent"], random_state=RANDOM_STATE)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

def build_pipe(clf):

    tfidf = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
    ])
    return Pipeline([("tfidf", tfidf), ("clf", clf)])

models = {
    "Naive Bayes":         MultinomialNB(alpha=0.1),
    "Logistic Regression": LogisticRegression(C=50, max_iter=3000),
    "SVM (linear)":        SVC(kernel="linear", C=5, probability=True, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
results = []
fitted = {}
for name, clf in models.items():
    pipe = build_pipe(clf)
    cv_f1 = cross_val_score(pipe, df["clean"], df["intent"], cv=cv, scoring="f1_macro").mean()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    results.append({
        "Model": name,
        "CV macro-F1": round(cv_f1, 3),
        "Test accuracy": round(accuracy_score(y_test, pred), 3),
        "Precision": round(precision_score(y_test, pred, average="macro", zero_division=0), 3),
        "Recall": round(recall_score(y_test, pred, average="macro", zero_division=0), 3),
        "F1-score": round(f1_score(y_test, pred, average="macro", zero_division=0), 3),
    })

results_df = pd.DataFrame(results).set_index("Model")

results_df["Selection score"] = ((results_df["CV macro-F1"] + results_df["F1-score"]) / 2).round(3)
results_df

## 6. Evaluation

Accuracy, precision, recall, F1 and the confusion matrix for the best model.

In [ ]:
best_name = results_df["Selection score"].idxmax()
best_model = fitted[best_name]
print("Best model:", best_name, "\n")

y_pred = best_model.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, average='macro', zero_division=0):.3f} (macro)")
print(f"Recall   : {recall_score(y_test, y_pred, average='macro', zero_division=0):.3f} (macro)")
print(f"F1-score : {f1_score(y_test, y_pred, average='macro', zero_division=0):.3f} (macro)\n")
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(df["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion matrix – {best_name}")
plt.tight_layout()
plt.show()

final_model = build_pipe(models[best_name].__class__(**models[best_name].get_params()))
final_model.fit(df["clean"], df["intent"])
KNOWN_WORDS = {w for text in df["clean"] for w in text.split()}

## 7. Knowledge base

One entry for each intent. The numbers follow commonly used guidelines (ADA, WHO) but everyone's targets are different, so every answer tells the user to follow their own doctor.

In [ ]:
knowledge_base = {
    "diabetes_basics": {
        "title": "About diabetes",
        "answer": (
            "Diabetes is a long-term condition in which the body cannot properly control blood glucose (sugar), "
            "either because it makes too little insulin or cannot use insulin well.\n"
            "• Type 1: an autoimmune condition; the pancreas makes little or no insulin, so insulin treatment is always needed.\n"
            "• Type 2: the most common type; the body resists insulin and/or makes too little. It is managed with healthy lifestyle, "
            "tablets and sometimes insulin.\n"
            "• Gestational: high glucose first found in pregnancy; it usually settles after delivery but raises later type 2 risk.\n"
            "• Prediabetes: glucose is above normal but below the diabetes range; lifestyle change can often prevent type 2."
        ),
    },
    "symptoms": {
        "title": "Symptoms and diagnosis",
        "answer": (
            "Common symptoms: increased thirst, frequent urination, extreme hunger, unexplained weight loss, tiredness, "
            "blurred vision, slow-healing cuts or frequent infections, and tingling in hands or feet. Type 2 diabetes may cause no symptoms for years.\n"
            "Diagnosis needs blood tests confirmed by a doctor. Commonly used cut-offs: fasting plasma glucose of 126 mg/dL or more, "
            "HbA1c of 6.5% or more, 2-hour glucose tolerance test of 200 mg/dL or more, or a random glucose of 200 mg/dL or more with symptoms.\n"
            "If you notice these symptoms, please arrange a test with a doctor."
        ),
    },
    "diet_nutrition": {
        "title": "Diet and nutrition",
        "answer": (
            "• Build a balanced plate: about half non-starchy vegetables, a quarter whole grains or other high-fibre carbohydrates, a quarter lean protein.\n"
            "• Choose high-fibre, minimally processed foods (whole grains, pulses, vegetables, nuts).\n"
            "• Limit sugary drinks, sweets and refined flour products; fruit is allowed in sensible portions.\n"
            "• Keep portion sizes and meal timings regular and watch total carbohydrates per meal.\n"
            "• Rice, roti and fruit are not banned; portion size, type (for example brown rice, whole wheat) and pairing with protein and vegetables matter.\n"
            "A registered dietitian can build a personal plan."
        ),
    },
    "medication_insulin": {
        "title": "Medication and insulin",
        "answer": (
            "• Metformin is commonly the first medicine for type 2 diabetes; other tablets and injectable medicines (such as DPP-4 inhibitors, SGLT2 inhibitors, GLP-1 medicines) may be added by a doctor.\n"
            "• Insulin comes as rapid-acting, short-acting, intermediate and long-acting types. People with type 1 diabetes always need insulin; some people with type 2 do too.\n"
            "• Take medicines exactly as prescribed, rotate injection sites (abdomen, thigh, upper arm), and store unopened insulin in the fridge and the pen in use at room temperature as the label says.\n"
            "• Watch for low blood sugar, and never change, skip or stop a dose without asking your doctor."
        ),
    },
    "glucose_monitoring": {
        "title": "Glucose monitoring",
        "answer": (
            "• Check glucose with a glucometer (finger prick) or a continuous glucose monitor (CGM). How often depends on your treatment; follow your doctor's advice.\n"
            "• Typical targets for many adults with diabetes: 80–130 mg/dL before meals and below 180 mg/dL one to two hours after starting a meal. Your doctor may set different targets.\n"
            "• HbA1c shows the average glucose over about 2–3 months; many adults aim for below 7%. It is usually tested every 3–6 months.\n"
            "• Keep a log, wash and dry hands before testing, and rotate fingertips."
        ),
    },
    "exercise_lifestyle": {
        "title": "Exercise and lifestyle",
        "answer": (
            "• Aim for about 150 minutes a week of moderate activity (such as brisk walking or cycling), spread over at least 3 days, plus 2 sessions of strength training.\n"
            "• Check glucose before and after exercise, carry fast-acting carbohydrate, wear proper footwear and drink water.\n"
            "• Good sleep, stress management, not smoking and a healthy weight all help glucose control; modest weight loss can greatly improve type 2 diabetes.\n"
            "• Speak to your doctor before starting a new exercise programme, especially if you have eye, nerve, kidney or heart problems."
        ),
    },
    "complications_prevention": {
        "title": "Complications and prevention",
        "answer": (
            "Long-term high glucose can damage the eyes (retinopathy), kidneys (nephropathy), nerves (neuropathy), heart and blood vessels, and feet.\n"
            "Prevention:\n"
            "• Keep glucose, blood pressure and cholesterol within the targets set by your doctor.\n"
            "• Get a dilated eye exam every year and kidney tests (urine albumin and eGFR) every year.\n"
            "• Check your feet daily for cuts, blisters or redness, and never walk barefoot.\n"
            "• Do not smoke, and keep up with regular check-ups and vaccinations."
        ),
    },
    "emergency": {
        "title": "Emergency care",
        "answer": (
            "• Low blood sugar (below 70 mg/dL), if you are awake and can swallow: use the 15-15 rule. Take 15 g of fast-acting sugar (glucose tablets or about half a cup of juice), "
            "recheck after 15 minutes, and repeat if still low. Then eat a snack or meal.\n"
            "• Severe low sugar (confusion, cannot swallow, seizure, unconscious): do not put food or drink in the mouth. Call emergency services immediately; if trained, give glucagon.\n"
            "• Very high sugar with vomiting, stomach pain, fruity breath, fast breathing or drowsiness may be diabetic ketoacidosis: go to a hospital emergency department immediately."
        ),
    },
}
assert set(knowledge_base) == set(corpus)
print("Knowledge base entries:", len(knowledge_base))

## 8. Rules

| Rule | What it does |
| --- | --- |
| R1 | Emergency escalation |
| R2 | Checks a glucose number in the question |
| R3 | Not about diabetes |
| R4 | Low confidence, asks to rephrase |
| R5 | Never advises on medicine doses |
| R6 | Never diagnoses |
| R7 | Empty input |
| R8 | Disclaimer on every answer |

In [ ]:
CONFIDENCE_THRESHOLD = 0.30

EMERGENCY_WORDS = [
    r"unconscious", r"not responding", r"unresponsive", r"passed out", r"fainted", r"seizure", r"convuls",
    r"can'?t wake", r"cannot wake", r"not breathing", r"chest pain", r"fruity breath", r"breath smells fruity",
    r"ketoacidosis", r"\bdka\b", r"very low sugar", r"severe hypoglyc", r"\bcoma\b",
]
OFF_TOPIC = [
    r"cancer", r"covid", r"corona", r"fracture", r"broken", r"asthma", r"migraine", r"headache", r"\bfever\b",
    r"\bflu\b", r"\bcold\b", r"dental", r"tooth", r"skin rash", r"\bhair\b", r"pregnan", r"thyroid", r"\bhiv\b",
    r"malaria", r"dengue", r"blood pressure", r"\bstock", r"weather", r"movie", r"cricket",
]
DOMAIN_TERMS = ["diabet", "sugar", "glucose", "insulin", "a1c", "metformin", "carb", "glycem", "ketoacidosis", "glucometer", "cgm"]
DOSE_WORDS = (r"(how (much|many) (units|insulin)|increase.*dose|decrease.*dose|reduce.*dose|skip.*(dose|insulin|tablet)|"
                r"stop (taking|my) (insulin|metformin|medicine|tablet)|change.*dose|how many units)")
DIAGNOSIS_WORDS = r"(do i have|am i (a )?diabetic|do you think i have|diagnose me|is it diabetes)"

EMERGENCY_MESSAGE = (
    "🚨 This may be a medical emergency. Call your local emergency number NOW (112 in India; 911 in the US) "
    "or go to the nearest hospital. Do not wait for an online answer."
)
DISCLAIMER = "ℹ️ General information only, not medical advice. Please follow your own doctor's plan."

def find_glucose_value(text):
    t = text.lower()
    m = re.search(r"(?:sugar|glucose|bg|reading)\D{0,15}(\d{2,3})\b", t) or re.search(r"\b(\d{2,3})\s*mg/?dl\b", t)
    return int(m.group(1)) if m else None

def has_domain_term(text):
    t = text.lower()
    return any(term in t for term in DOMAIN_TERMS)

def is_out_of_scope(text):
    t = text.lower()
    return any(re.search(p, t) for p in OFF_TOPIC) and not has_domain_term(text)

## 9. Putting it all together

Cleaning, intent classification, rules, concept-map reasoning and knowledge lookup all happen inside one class.

In [ ]:
def related_facts(query, max_facts=3):
    q = query.lower()
    mentioned = [c for c, aliases in concepts.items() if any(a in q for a in aliases)]
    facts = []
    for a, rel, b in relationships:
        if a in mentioned or b in mentioned:
            facts.append(f"{a} {rel} {b}.")
    return mentioned, facts[:max_facts]

def classify(query):
    clean = preprocess(query)
    probs = final_model.predict_proba([clean])[0]
    classes = final_model.classes_
    order = np.argsort(probs)[::-1]
    return classes[order[0]], float(probs[order[0]]), [(classes[i], float(probs[i])) for i in order[:3]]

class DiabetesAssistant:
    def respond(self, query):
        out = {"query": query, "intent": None, "confidence": None, "rules": [], "concepts": [], "text": ""}
        parts = []

        if not query or not query.strip():
            out["rules"].append("R7 empty-input")
            out["text"] = "Please type a question about diabetes, for example: 'What foods should I avoid?'"
            return out

        tokens = preprocess(query, as_tokens=True)
        intent, conf, top3 = classify(query)
        out["confidence"], out["top3"] = round(conf, 3), top3
        ql = query.lower()
        emergency = any(re.search(p, ql) for p in EMERGENCY_WORDS)
        dose_flag = bool(re.search(DOSE_WORDS, ql))
        diagnosis_flag = bool(re.search(DIAGNOSIS_WORDS, ql))

        value = find_glucose_value(query)
        if value is not None:
            out["rules"].append(f"R2 glucose-value ({value} mg/dL)")
            if value < 54:
                emergency = True
                parts.append(f"A reading of {value} mg/dL is dangerously low (below 54 mg/dL).")
            elif value < 70:
                parts.append(f"{value} mg/dL is low (below 70 mg/dL). Follow the 15-15 rule now: take 15 g of fast-acting sugar and recheck in 15 minutes.")
                intent = "emergency"
            elif value <= 180:
                parts.append(f"{value} mg/dL is within or close to common target ranges (80–130 before meals, under 180 after meals). Your own targets may differ.")
            elif value < 300:
                parts.append(f"{value} mg/dL is high. Drink water, follow your doctor's plan, and contact your doctor if readings stay high.")
            else:
                parts.append(f"{value} mg/dL is very high. Contact your doctor today; if you have vomiting, stomach pain or fast breathing, go to hospital.")

        if emergency:
            out["rules"].append("R1 emergency-escalation")
            intent = "emergency"
            parts.insert(0, EMERGENCY_MESSAGE)

        if not emergency and is_out_of_scope(query):
            out["rules"].append("R3 unsupported-query")
            out["intent"] = "unsupported"
            out["text"] = ("I can only help with diabetes care information (types, symptoms, diet, medicines, monitoring, "
                           "exercise, complications and emergencies). For other health problems please consult a doctor.\n\n" + DISCLAIMER)
            return out

        known = any(tok in KNOWN_WORDS for tok in tokens)
        if not emergency and value is None and not dose_flag and not diagnosis_flag and (conf < CONFIDENCE_THRESHOLD or not known):
            out["rules"].append("R4 low-confidence")
            out["intent"] = "low_confidence"
            out["text"] = (f"I am not sure what you mean (confidence {conf:.0%}). Could you rephrase or be more specific?\n"
                           "I can help with: " + ", ".join(sub for subs in taxonomy.values() for sub in subs) + ".\n\n" + DISCLAIMER)
            return out

        if dose_flag:
            out["rules"].append("R5 dose-safety")
            parts.append("⚠️ I cannot recommend or change medicine doses. The right dose depends on your weight, glucose readings and other medicines, "
                         "so please ask your doctor or diabetes educator before changing, skipping or stopping any medicine.")
            intent = "medication_insulin"

        if diagnosis_flag:
            out["rules"].append("R6 no-diagnosis")
            parts.append("⚠️ I cannot diagnose you. Only a doctor can confirm diabetes using blood tests (fasting glucose, HbA1c or a glucose tolerance test).")
            intent = "symptoms"

        entry = knowledge_base[intent]
        out["intent"] = intent
        cat, sub = intent_to_path[intent]
        out["path"] = f"{cat} › {sub}"
        mentioned, facts = related_facts(query)
        out["concepts"] = mentioned
        parts.append(f"{entry['title']}:\n{entry['answer']}")
        if facts:
            parts.append("Related facts (concept map): " + " ".join(facts))
        out["rules"].append("R8 disclaimer")
        parts.append(DISCLAIMER)
        out["text"] = "\n\n".join(parts)
        return out

assistant = DiabetesAssistant()

def show(query):
    r = assistant.respond(query)
    print("=" * 90)
    print("USER      :", query)
    print("INTENT    :", r["intent"], "| confidence:", r["confidence"], "| path:", r.get("path", "-"))
    print("RULES     :", ", ".join(r["rules"]) or "none")
    if r["concepts"]:
        print("CONCEPTS  :", ", ".join(r["concepts"]))
    print("ASSISTANT :\n" + r["text"])

## 10. Testing

The five test types the assignment asks for.

**Test 1: one query for each intent**

In [ ]:
test_each_intent = [
    "Can you tell me what type 1 diabetes means?",
    "I keep feeling thirsty and visiting the toilet at night. Is that a symptom?",
    "Is brown rice better than white rice for sugar patients?",
    "What are the side effects of metformin?",
    "What is the ideal HbA1c target?",
    "Is 30 minutes of brisk walking daily enough?",
    "How does high sugar harm the kidneys?",
    "What do I do if my blood sugar drops very low and I feel shaky?",
]
for q in test_each_intent:
    show(q)

**Test 2: differently worded queries**

In [ ]:
show("What kinds of food are bad for someone with diabetes?")
show("How is type 1 different from type 2 diabetes?")

**Test 3: unsupported healthcare queries**

In [ ]:
show("How do I treat a broken arm?")
show("What is the best medicine for migraine headaches?")

**Test 4: low-confidence queries**

In [ ]:
show("What should I do about it?")
show("Is my condition serious?")

**Test 5: emergency queries**

In [ ]:
show("My father is diabetic and he is unconscious and not responding!")
show("My blood sugar is 42 and I feel confused")

**Extra: dose and diagnosis rules**

In [ ]:
show("How many units of insulin should I take tonight?")
show("I am always thirsty, do I have diabetes?")

## 11. Chat box

Run the cell, type a question and press Ask. You can ask as many as you want.

In [ ]:
def launch_ui():
    try:
        import ipywidgets as widgets
        from IPython.display import display, HTML
    except Exception:
        print("ipywidgets not available – use chat_loop() instead.")
        return
    box = widgets.Text(placeholder="Ask a question about diabetes…", layout=widgets.Layout(width="70%"))
    btn = widgets.Button(description="Ask", button_style="primary")
    clear = widgets.Button(description="Clear")
    out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="8px", max_height="480px", overflow_y="auto"))

    def on_ask(_=None):
        q = box.value.strip()
        with out:
            if not q:
                print("Please type a question.")
                return
            r = assistant.respond(q)
            print("You      :", q)
            print(f"[intent: {r['intent']} | confidence: {r['confidence']} | rules: {', '.join(r['rules'])}]")
            print("Assistant:", r["text"], "\n" + "-" * 80)
        box.value = ""

    btn.on_click(on_ask)
    try:
        box.on_submit(on_ask)
    except Exception:
        pass
    clear.on_click(lambda _: out.clear_output())
    display(widgets.HTML("<h3>🩺 Diabetes Care Information Assistant</h3>"),
            widgets.HBox([box, btn, clear]), out)

def chat_loop():
    print("Diabetes Care Information Assistant – type 'exit' to stop.")
    while True:
        q = input("\nYou: ")
        if q.strip().lower() in {"exit", "quit"}:
            print("Goodbye! Stay healthy.")
            break
        r = assistant.respond(q)
        print(f"[intent: {r['intent']} | confidence: {r['confidence']} | rules: {', '.join(r['rules'])}]")
        print("Assistant:", r["text"])

launch_ui()

## 12. Summary

* Cleaning: lowercase, tokens, stopwords, lemmas
* Intent: TF-IDF plus the best of Naive Bayes, Logistic Regression and SVM
* Reasoning: 8 rules and a concept map
* Retrieval: one knowledge base entry per intent
* Limits: only 160 training queries, so the accuracy is just a rough idea. It cannot diagnose or prescribe.